# **ANALISIS SPASIOTEMPORAL DIVERGENSI HOTSPOT & COLDSPOT (Rentang Waktu: 2010 - 2020)**

In [ ]:
pip install pandas numpy matplotlib openpyxl

In [ ]:
# 1. IMPOR PUSTAKA & PENGATURAN LINGKUNGAN
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
import warnings
from google.colab import drive

# Mengabaikan peringatan (warnings) agar output Colab tetap bersih
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# 2. SETUP DIREKTORI & PEMUATAN DATA GABUNGAN
# ==============================================================================
# Tentukan path direktori utama penyimpanan data CSV
base_path = '/content/drive/My Drive/Colab Notebooks/Skripsi/NASA-FIRMS Fire (New2)/Titik Panas 2010-2020/'

# Buat direktori khusus untuk menyimpan hasil ekspor gambar grafik
output_folder_img = os.path.join(base_path, 'Hasil_Visualisasi_Per_Tahun')
os.makedirs(output_folder_img, exist_ok=True)
print(f"[INFO] Direktori penyimpanan grafik: {output_folder_img}")

# --- Proses Iterasi dan Penggabungan Data (2010-2020) ---
list_data = []
print("\n[INFO] Menginisiasi pemuatan data historis 2010-2020...")

for tahun in range(2010, 2021):
    nama_file = f'Hotspot_{tahun}_Hutan_Adm_Itc.csv'
    full_path = os.path.join(base_path, nama_file)

    if os.path.exists(full_path):
        try:
            df_temp = pd.read_csv(full_path)
            df_temp['Tahun'] = tahun # Menambahkan penanda temporal
            list_data.append(df_temp)
            print(f"   [OK] Tahun {tahun} dimuat ({len(df_temp)} observasi).")
        except Exception as e:
            print(f"   [ERROR] Gagal membaca {nama_file}: {e}")
    else:
        print(f"   [SKIP] File tidak ditemukan: {nama_file}")

# Validasi keberhasilan kompilasi data
if not list_data:
    raise SystemExit("[STOP] Eksekusi dihentikan: Tidak ada data yang ditemukan. Periksa path direktori.")

df_all = pd.concat(list_data, ignore_index=True)
print(f"\n[SUKSES] Total observasi tergabung: {len(df_all)} titik.")

In [ ]:
# ==============================================================================
# 3. KONFIGURASI ATRIBUT, LABEL, DAN PALET WARNA
# ==============================================================================
# Pemetaan nilai Gi_Bin (Z-Score) ke dalam kelas signifikansi akademis
label_map = {
    3: 'Hot Spot 99% Confidence',
    2: 'Hot Spot 95% Confidence',
    1: 'Hot Spot 90% Confidence',
    0: 'Not Significant',
   -1: 'Cold Spot 90% Confidence',
   -2: 'Cold Spot 95% Confidence',
   -3: 'Cold Spot 99% Confidence'
}
df_all['Label_Gi'] = df_all['Gi_Bin'].map(label_map)

# Definisi skema warna visualisasi (Pendekatan Thermal: Biru ke Merah)
colors = {
    'Cold Spot 99% Confidence': '#313695',
    'Cold Spot 95% Confidence': '#4575b4',
    'Cold Spot 90% Confidence': '#abd9e9',
    'Hot Spot 90% Confidence': '#fdae61',
    'Hot Spot 95% Confidence': '#d7191c',
    'Hot Spot 99% Confidence': '#a50026'
}

# Hierarki penumpukan (stacking) batang pada grafik visual
urutan_stack = [
    'Cold Spot 90% Confidence', 'Cold Spot 95% Confidence', 'Cold Spot 99% Confidence',
    'Hot Spot 90% Confidence', 'Hot Spot 95% Confidence', 'Hot Spot 99% Confidence'
]

# Hierarki kolom untuk ekspor data tabular (Excel)
urutan_excel = [
    'Cold Spot 99% Confidence', 'Cold Spot 95% Confidence', 'Cold Spot 90% Confidence',
    'Not Significant',
    'Hot Spot 90% Confidence', 'Hot Spot 95% Confidence', 'Hot Spot 99% Confidence'
]

# Ekstraksi Master List agar sumbu Y pada grafik selalu konsisten melintasi waktu
df_signifikan = df_all[df_all['Gi_Bin'] != 0]
master_adm = sorted(df_signifikan['WADMKK'].dropna().unique())
master_hutan = sorted(df_signifikan['Prbh_Fungs'].dropna().unique())

In [ ]:
# ==============================================================================
# 4. TABULASI STATISTIK DAN EKSPOR EXCEL
# ==============================================================================
print("\n[INFO] Mengkompilasi rekapitulasi tabular (Excel)...")

# A. Tabulasi Silang Wilayah Administrasi vs Tahun
tab_kab_tahun = pd.crosstab([df_all['WADMKK'], df_all['Tahun']], df_all['Label_Gi'])
tab_kab_tahun = tab_kab_tahun[[c for c in urutan_excel if c in tab_kab_tahun.columns]]

# B. Tabulasi Silang Fungsi Hutan vs Tahun
tab_hutan_tahun = pd.crosstab([df_all['Prbh_Fungs'], df_all['Tahun']], df_all['Label_Gi'])
tab_hutan_tahun = tab_hutan_tahun[[c for c in urutan_excel if c in tab_hutan_tahun.columns]]

# C. Matriks Tren Spasiotemporal (Khusus Hotspot)
df_hot_only = df_all[df_all['Gi_Bin'] > 0]
matriks_tren = pd.pivot_table(df_hot_only, index='WADMKK', columns='Tahun', values='SOURCE_ID', aggfunc='count', fill_value=0)

# Proses penulisan dan penyimpanan Workbook Excel
output_excel = os.path.join(base_path, 'Hasil_Analisis_Hotspot_2010_2020.xlsx')
try:
    with pd.ExcelWriter(output_excel) as writer:
        tab_kab_tahun.to_excel(writer, sheet_name='Detail_Kabupaten_Tahun')
        tab_hutan_tahun.to_excel(writer, sheet_name='Detail_Hutan_Tahun')
        matriks_tren.to_excel(writer, sheet_name='Matriks_Tren_Hotspot')
    print(f"   [OK] Laporan Excel tersimpan: {output_excel}")
except Exception as e:
    print(f"   [ERROR] Kegagalan kompilasi Excel: {e}")

In [ ]:
# ==============================================================================
# 5. MODUL VISUALISASI: DIVERGING STACKED BAR CHART
# ==============================================================================
def buat_grafik_tahunan(df_input, tahun_target, kolom_kategori, master_index, suffix_judul):
    """
    Menghasilkan grafik diverging bar yang merepresentasikan kontras
    frekuensi spasial antara zona panas (Hotspot) dan zona dingin (Coldspot).
    """
    # Menyaring data untuk menghilangkan observasi non-signifikan
    df_viz = df_input[df_input['Gi_Bin'] != 0].copy()
    pivot = pd.crosstab(df_viz[kolom_kategori], df_viz['Label_Gi'])

    # Memastikan konsistensi dimensi matriks untuk plotting
    for col in urutan_stack:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot = pivot.reindex(master_index, fill_value=0).sort_index(ascending=True)

    # Inisialisasi Kanvas Plotting
    fig, ax = plt.subplots(figsize=(14, 10))
    y_pos = np.arange(len(pivot))

    def pasang_label(rects, data_values, warna_teks):
        """Fungsi internal untuk menyematkan anotasi teks pada tengah batang grafik."""
        label_text = [str(int(abs(x))) if x != 0 else '' for x in data_values]
        ax.bar_label(rects, labels=label_text, label_type='center', color=warna_teks, fontsize=8)

    # --- PLOT SISI KANAN (HOT SPOTS) ---
    h90, h95, h99 = pivot['Hot Spot 90% Confidence'], pivot['Hot Spot 95% Confidence'], pivot['Hot Spot 99% Confidence']

    r1 = ax.barh(y_pos, h90, label='Hot Spot 90% Confidence', color=colors['Hot Spot 90% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r1, h90, 'black')

    r2 = ax.barh(y_pos, h95, left=h90, label='Hot Spot 95% Confidence', color=colors['Hot Spot 95% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r2, h95, 'white')

    r3 = ax.barh(y_pos, h99, left=h90+h95, label='Hot Spot 99% Confidence', color=colors['Hot Spot 99% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r3, h99, 'white')

    # --- PLOT SISI KIRI (COLD SPOTS - Arah Negatif) ---
    c90, c95, c99 = pivot['Cold Spot 90% Confidence'], pivot['Cold Spot 95% Confidence'], pivot['Cold Spot 99% Confidence']

    r4 = ax.barh(y_pos, -c90, label='Cold Spot 90% Confidence', color=colors['Cold Spot 90% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r4, c90, 'black')

    r5 = ax.barh(y_pos, -c95, left=-c90, label='Cold Spot 95% Confidence', color=colors['Cold Spot 95% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r5, c95, 'white')

    r6 = ax.barh(y_pos, -c99, left=-(c90+c95), label='Cold Spot 99% Confidence', color=colors['Cold Spot 99% Confidence'], edgecolor='white', linewidth=0.5)
    pasang_label(r6, c99, 'white')

    # --- FORMATTING TAMPILAN GRAFIK ---
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pivot.index, fontsize=11)
    ax.invert_yaxis() # Resolusi hierarki pembacaan (A-Z dari atas ke bawah)
    ax.axvline(0, color='black', linewidth=0.8) # Sumbu vertikal ekuilibrium

    # Mengubah label Sumbu X absolut agar tidak menampilkan nilai minus pada sisi Coldspot
    ticks = ax.get_xticks()
    ax.xaxis.set_major_locator(ticker.FixedLocator(ticks))
    ax.set_xticklabels([f'{int(abs(x))}' for x in ticks])

    # Konfigurasi Anotasi dan Grid
    plt.title(f'Divergensi Spasial Titik Panas Tahun {tahun_target}\n({suffix_judul})', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Jumlah Observasi (Frekuensi)', fontsize=11, fontweight='semibold')
    plt.grid(axis='x', linestyle='--', alpha=0.3)

    # Konstruksi Legenda (Dikelompokkan secara hierarkis)
    handles, labels = ax.get_legend_handles_labels()
    order = [2, 1, 0, 3, 4, 5]
    leg = ax.legend([handles[idx] for idx in order], [labels[idx] for idx in order],
              loc='upper right', ncol=1, fontsize=9, frameon=False, title="Tingkat Kepercayaan")
    leg._legend_box.align = "left"
    leg.get_title().set_fontweight('semibold')

    plt.tight_layout()

    # --- PROSES EXPORT GAMBAR ---
    nama_file_save = f"{suffix_judul.replace(' ', '_')}_{tahun_target}.png"
    path_save = os.path.join(output_folder_img, nama_file_save)

    plt.savefig(path_save, dpi=300, bbox_inches='tight')
    plt.close() # Membersihkan buffer memori grafik
    print(f"      [v] Diekspor: {nama_file_save}")

In [ ]:
# ==============================================================================
# 6. EKSEKUSI ITERASI VISUALISASI
# ==============================================================================
print("\n[INFO] Memulai perenderan grafik resolusi tinggi (2010-2020)...")

for thn in range(2010, 2021):
    print(f"   -> Memproses data grafis tahun {thn}...")
    df_curr = df_all[df_all['Tahun'] == thn]

    if df_curr.empty:
        print(f"      [!] Tidak ada observasi spasial di tahun {thn}. Dilewati.")
        continue

    # Pemanggilan modul fungsi grafik
    buat_grafik_tahunan(df_curr, thn, 'WADMKK', master_adm, 'Wilayah Administrasi')
    buat_grafik_tahunan(df_curr, thn, 'Prbh_Fungs', master_hutan, 'Kawasan Hutan')

print(f"\n[SELESAI] Seluruh alur komputasi telah tuntas dieksekusi.")